In [23]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD

In [18]:
#datasets

x = [[1,2],[3,4],[5,6],[7,8]]
y = [[3],[7],[11],[15]]

#convertendo para tensores
X = torch.tensor(x).float()
Y = torch.tensor(y).float()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
X = X.to(device)
Y = Y.to(device)

In [19]:
#criando o dataset para uso com o PyTorch

class MyDataset(Dataset):
    def __init__(self,x,y):
        self.x = torch.tensor(x).float().to(device)
        self.y = torch.tensor(y).float().to(device)
    def __len__(self):
        return len(self.x)
    def __getitem__(self,ix):
        return self.x[ix], self.y[ix]

In [20]:
ds = MyDataset(x,y)

In [28]:
#carregando os dados em batches

d1 = DataLoader(ds, batch_size=3, shuffle=True)

In [22]:
for x,y in d1:
    print(x,y)

tensor([[5., 6.],
        [7., 8.],
        [1., 2.]], device='cuda:0') tensor([[11.],
        [15.],
        [ 3.]], device='cuda:0')
tensor([[3., 4.]], device='cuda:0') tensor([[7.]], device='cuda:0')


In [24]:
#criando a rede neural

class MyNeuralNet(nn.Module):
  def __init__(self):    
    super().__init__()
    self.layer1 = nn.Linear(2,8)
    self.activation = nn.ReLU()
    self.layer2 =  nn.Linear(8,1)

  def forward(self,x):
    x = self.layer1(x)
    x = self.activation(x)
    x = self.layer2(x)
    return x

In [25]:
#criando o modelo

model = MyNeuralNet()
model = model.to(device)

#definindo a loss function e o otimizador

loss_func = nn.MSELoss()
opt = SGD(model.parameters(), lr=0.001)

In [27]:
losses = []

#treinando por 50 épocas

for _ in range(50):
    for data in d1:
        opt.zero_grad() #configura os gradientes para 0
        x1,y1 = data
        loss_value = loss_func(model(x1),y1)
        loss_value.backward()

        opt.step()
        losses.append(loss_value.cpu().detach().numpy())